# Build Step 3 — FastAPI Upload API (Direct-to-Volume Flow)

**Framework:** File Validation & Routing (`fvr`) • **Deliverable:** the upload API — manifest → streamed parts → complete/finalize — writing bytes **directly to the staging volume** (Storage Addendum; GridFS deprecated) • **Checkpoint:** feeds **R2** (rule dry-run + load test) once Step 4's worker exists

## 1. What Step 3 covers (and what it does not)

| In scope (this API) | Out of scope (later steps) |
| --- | --- |
| `POST /upload-sessions` — manifest + pre-staging FILE-rule checks | Validation worker — HEADER/CONTENT rules, routing, volume moves (**Step 4**) |
| `PUT /upload-sessions/{sid}/files/{fid}` — streamed upload w/ on-the-fly sha256 | Bronze ingestion callbacks (Phase 2) |
| `POST /upload-sessions/{sid}/complete` — finalize: registry + lifecycle + `FILE_ARRIVED` | OPS APIs — re-classify, re-validate, purge (Phase 3) |
| Files API client (PUT/HEAD/GET/DELETE) + M2M auth | Portal UI (React) |

## 2. Prerequisites (must be live before the first upload)

1. **Volumes — ✅ created & smoke-tested (2026-11-09):** `mdd_platform.fvr` schema with managed volumes `staging`, `intake`, `quarantine` → roots `/Volumes/mdd_platform/fvr/{staging,intake,quarantine}`.
2. **MongoDB replica set** with Steps 1–2 applied: 7 registry collections seeded (volume_map now carries the **real roots**), 8 operational collections + indexes in place.
3. **Service principal** (Databricks, M2M) granted **`WRITE FILES`** on `mdd_platform.fvr.staging` (the API only ever writes staging; intake/quarantine writes belong to the Step-4 worker). For dev, a PAT in a vault secret works.
4. **Python service skeleton:** `fastapi`, `uvicorn`, `motor` (async Mongo), `httpx`, `pydantic-settings`; package layout below.

```text
fvr_api/
  main.py              # FastAPI app, middleware, exception map
  config.py            # pydantic Settings — hosts, tokens, roots
  clients/databricks.py # Files API client (PUT/HEAD/GET/DELETE + M2M OAuth)
  clients/mongo.py      # motor client, collections, txn helpers
  api/sessions.py      # the three endpoints (this notebook's core)
  domain/ids.py        # file_/session_/event_ id generators
  domain/rules_pre.py  # pre-staging FILE-rule checks (registry-driven)
```

> Every write below uses the guarded-atomic patterns from the Resilience Addendum — the MongoDB validators created in Step 2 reject anything malformed at the door. Code is condensed for review; file-level docstrings and typing live in the service repo.

## 3. `config.py` + `clients/databricks.py` — Files API client (M2M)

Roots come from the **registry** (`volume_map`), not env vars — one source of truth, changed by document edit:

```python
# config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    databricks_host: str                       # https://dbc-….cloud.databricks.com
    databricks_client_id: str | None = None    # M2M service principal (vault-backed)
    databricks_client_secret: str | None = None
    databricks_token: str | None = None         # dev fallback: PAT from vault
    mongo_uri: str                              # mongodb+srv://… (replica set)
    staging_root: str = "/Volumes/mdd_platform/fvr/staging"  # cross-checked vs volume_map at startup
settings = Settings()
```

```python
# clients/databricks.py — the ONLY module that talks to Databricks
import asyncio, httpx, time
from .config import settings

class FilesAPIClient:
    """Thin async wrapper over /api/2.0/fs — streaming, M2M OAuth, retry on 429/5xx."""

    def __init__(self):
        self._token, self._token_exp = None, 0.0
        self._http = httpx.AsyncClient(base_url=settings.databricks_host, timeout=120.0)

    async def _auth_headers(self) -> dict:
        if settings.databricks_client_id:                       # M2M OAuth (prod path)
            if time.time() > self._token_exp - 60:
                r = await self._http.post("/oidc/v1/token", data={
                    "grant_type": "client_credentials",
                    "client_id": settings.databricks_client_id,
                    "client_secret": settings.databricks_client_secret,
                    "scope": "all-apis"})
                r.raise_for_status()
                body = r.json()
                self._token, self._token_exp = body["access_token"], time.time() + body["expires_in"]
            return {"Authorization": f"Bearer {self._token}"}
        return {"Authorization": f"Bearer {settings.databricks_token}"}   # dev PAT

    async def _retry(self, method: str, path: str, **kw) -> httpx.Response:
        for attempt in range(5):                               # backoff: 1s, 2s, 4s, 8s
            try:
                r = await self._http.request(method, path, headers=await self._auth_headers(), **kw)
            except httpx.RequestError:                         # transport faults retry too (review F1)
                await asyncio.sleep(2 ** attempt); continue
            if r.status_code not in (429, 500, 502, 503, 504):
                return r
            await asyncio.sleep(2 ** attempt)
        return r

    # ---- the four primitives (paths are /Volumes/… absolute) ----
    async def put_file(self, path: str, stream) -> httpx.Response:
        """Streaming PUT — overwrite defaults to TRUE: an interrupted upload is
        safely re-PUT (last write wins). That IS our resume story for <15 MB files."""
        return await self._retry("PUT", f"/api/2.0/fs/files{path}", content=stream)

    async def head(self, path: str) -> httpx.Response:
        return await self._retry("HEAD", f"/api/2.0/fs/files{path}")

    async def get_file(self, path: str) -> httpx.Response:
        return await self._retry("GET", f"/api/2.0/fs/files{path}")

    async def delete_file(self, path: str) -> httpx.Response:
        return await self._retry("DELETE", f"/api/2.0/fs/files{path}")
```

> **Startup cross-check:** on boot, load `volume_map` from Mongo and assert the `STAGING` row's `volume_root` matches `settings.staging_root` — a config/registry drift fails fast with a 503, never silently uploads to the wrong volume.

## 4. `api/sessions.py` — the three endpoints

```python
# api/sessions.py (condensed — error mapping via exception handlers in main.py)
import hashlib, uuid
from datetime import datetime, timezone
from pymongo.errors import DuplicateKeyError
from fastapi import APIRouter, Request, HTTPException
from ..clients import mongo, databricks as dbx
from ..domain.rules_pre import pre_staging_checks      # registry-driven FILE-rule checks

router = APIRouter(prefix="/upload-sessions", tags=["upload"])

# ---------- POST /upload-sessions : manifest ----------
@router.post("", status_code=201)
async def create_session(manifest: dict, request: Request):
    """Pre-staging FILE rules run HERE — a bad file is rejected before a single
    byte is written to the staging volume (Storage Addendum §3)."""
    files = manifest["files"]                                  # [{original_name, declared_size, file_type_code?}]
    ft_code = manifest.get("file_type_code")
    ft = await pre_staging_checks(ft_code, files)             # type allowed? ext allowed? size<=max? >0?
    #   -> raises 400 (bad ext) / 413 (size) / 400 (empty) — all rule-evidenced, NOT yet persisted

    sid = "ses_" + uuid.uuid4().hex[:16]
    parts = [{"file_id": "file_" + uuid.uuid4().hex[:16],
              "staged_path": f"{dbx.staging_root()}/sessions/{sid}/{f['original_name']}",
              "status": "UPLOADING", "bytes_received": 0} for f in files]
    await mongo.db.upload_sessions.insert_one({
        "session_id": sid, "user_id": request.state.user_id,
        "roles_snapshot": request.state.roles, "source_id": "src_portal",
        "file_type_id": ft["file_type_id"],
        "files_declared_count": len(files), "files_received_count": 0,
        "parts": parts, "status": "IN_PROGRESS",
        "started_at": datetime.now(timezone.utc), "completed_at": None})
    return {"session_id": sid,
            "files": [{"file_id": p["file_id"], "upload_url":
                        f"/upload-sessions/{sid}/files/{p['file_id']}"} for p in parts]}

# ---------- PUT /upload-sessions/{sid}/files/{fid} : streamed upload ----------
@router.put("/{sid}/files/{fid}")
async def upload_part(sid: str, fid: str, request: Request):
    """One streaming pass: bytes flow client -> FastAPI -> staging volume,
    sha256 and byte-count computed on the fly (never buffered whole)."""
    sess = await mongo.db.upload_sessions.find_one(
        {"session_id": sid, "parts.file_id": fid, "status": "IN_PROGRESS"})
    if not sess:
        raise HTTPException(409, "session not IN_PROGRESS or unknown file")  # state guard
    part = next(p for p in sess["parts"] if p["file_id"] == fid)
    ft = await mongo.db.file_types.find_one({"file_type_id": sess["file_type_id"]})
    max_bytes = ft["max_file_size_mb"] * 1024 * 1024

    sha, received, cap_exceeded = hashlib.sha256(), 0, False

    async def gen():                       # chunk-for-chunk: hash, cap-check, forward
        nonlocal sha, received, cap_exceeded
        async for chunk in request.stream():
            received += len(chunk)
            if received > max_bytes:
                cap_exceeded = True        # abort flag — raising INSIDE a generator consumed
                return                     # by httpx surfaces as an opaque send error, not a 413
            sha.update(chunk)
            yield chunk

    r = await dbx.client.put_file(part["staged_path"], gen())   # overwrite=true → idempotent re-PUT
    if cap_exceeded:                       # review F4: kill the partial object, THEN reject
        await dbx.client.delete_file(part["staged_path"])
        raise HTTPException(413, f"exceeds {ft['max_file_size_mb']}MB (FILE_SIZE_MAX)")
    r.raise_for_status()                                        # 4xx/5xx after retries → 502 to client

    await mongo.db.upload_sessions.update_one(                  # record progress (no finalize yet)
        {"session_id": sid, "parts.file_id": fid, "status": "IN_PROGRESS"},   # review F5: guarded
        {"$set": {"parts.$.bytes_received": received, "parts.$.sha256": sha.hexdigest()}})
    return {"file_id": fid, "bytes_received": received, "sha256": sha.hexdigest(),
            "finalized": False}

# ---------- POST /upload-sessions/{sid}/complete : finalize ----------
@router.post("/{sid}/complete")
async def complete_session(sid: str):
    """Ack-after-durable-write (C6): success ONLY after every part is verified
    on the volume AND the Mongo txn (registry + lifecycle + event) has committed."""
    sess = await mongo.db.upload_sessions.find_one(              # review F5: state guard —
        {"session_id": sid, "status": "IN_PROGRESS"})            # a completed session cannot re-finalize
    if not sess:
        raise HTTPException(409, "session unknown or not IN_PROGRESS")
    finalized, dups = [], []
    for part in sess["parts"]:
        h = await dbx.client.head(part["staged_path"])          # volume truth: exists + size
        if h.status_code != 200 or int(h.headers["Content-Length"]) != part["bytes_received"]:
            raise HTTPException(422, {"file_id": part["file_id"],
                                      "detail": "staged file missing or size mismatch — re-upload the part"})
        try:
            async with await mongo.client.start_session() as ms:
                async with ms.start_transaction():
                    await mongo.db.file_registry.insert_one(ms, {   # unique sha256 → 409 (C1)
                        "file_id": part["file_id"], "source_id": sess["source_id"],
                        "original_name": part["staged_path"].rsplit("/", 1)[1],
                        "staged_path": part["staged_path"], "gridfs_file_id": None,
                        "size_bytes": part["bytes_received"],
                        "extension": "." + part["staged_path"].rsplit(".", 1)[1],
                        "sha256_checksum": part["sha256"], "current_status": "RECEIVED",
                        "upload_session_id": sid, "uploaded_by": sess["user_id"],
                        "retry_count": 0, "received_at": datetime.now(timezone.utc),
                        "updated_at": datetime.now(timezone.utc),
                        "volume_purged_at": None, "legal_hold": None})
                    await mongo.db.file_lifecycle.insert_one(ms, {   # first location entry
                        "event_id": "flc_" + uuid.uuid4().hex[:16],
                        "file_id": part["file_id"], "location": "STAGING_VOLUME",
                        "timestamp": datetime.now(timezone.utc), "triggered_by_job": None})
                    await mongo.db.event_log.insert_one(ms, {        # dedup key (C2): exactly once
                        "event_id": "evt_" + uuid.uuid4().hex[:16],
                        "event_type": "FILE_ARRIVED", "source_id": sess["source_id"],
                        "dedup_key": part["file_id"], "status": "RECEIVED",
                        "correlation_id": part["file_id"], "timestamp": datetime.now(timezone.utc),
                        "payload": {"file_id": part["file_id"], "session_id": sid}})
        except DuplicateKeyError:                               # review F2: pymongo error, not motor's
            dups.append(part["file_id"])                          # same sha256 seen before
        finalized.append(part["file_id"])

    done = len(finalized) - len(dups)
    await mongo.db.upload_sessions.update_one(                   # terminal state (guarded)
        {"session_id": sid, "status": "IN_PROGRESS"},
        {"$set": {"status": "COMPLETED_PARTIAL" if dups or done < sess["files_declared_count"]
                  else "COMPLETED", "files_received_count": done, "completed_at": datetime.now(timezone.utc)}})
    return {"session_id": sid, "finalized": finalized,
            "duplicates": dups, "status": "COMPLETED" if not dups else "COMPLETED_PARTIAL"}
```

> The **worker (Step 4)** reacts to `FILE_ARRIVED` on the change stream, downloads the staged file (re-hashing to verify sha256 in transit), runs HEADER/CONTENT rules, routes via `routing_policies` + `volume_map` to intake/quarantine, and deletes the staging copy only after the destination PUT is verified — none of that lives in this API.

## 5. Error map, resilience & operational hooks

| Client sees | Trigger | System state afterwards | Safe retry? |
| --- | --- | --- | --- |
| `400` rule-evidenced | FILE_TYPE_ALLOWED / FILE_NOT_EMPTY (manifest) | nothing written — rejected pre-staging | no (fix file) |
| `413` | declared or streamed size > max (FILE_SIZE_MAX) | partial PUT possible on staging — orphan, janitor cleans | no (fix file) |
| `409` unknown file / session | part uploaded against finalized/abandoned session (state guard) | nothing changed | no — request new manifest |
| `409` duplicates list in complete | same sha256 finalized before (C1) | FIRST copy intact; dup is recorded, bytes not re-registered | yes (get original file_id) |
| `422` size mismatch | HEAD shows staged size ≠ streamed count — interrupted PUT | part not finalized; re-PUT is idempotent (overwrite=true) | **yes — re-upload the part** |
| `502/503` | Files API 429/5xx exhausted 5 retries | no registry row — invisible until re-upload | **yes — whole flow is idempotent** |

**Resilience patterns in play** (traceability to the Resilience Addendum):

* **C2/C6 ack-after-durable-write** — the upload 200 means the PUT landed; the complete 200 means registry+lifecycle+event are **in one committed transaction**. Anything between is re-drivable: re-PUT (idempotent), re-complete (HEAD re-verifies, FILE_ARRIVED dedup key blocks double-fire).
* **Janitor hook** — sessions stuck `IN_PROGRESS` past TTL are marked `ABANDONED` and their whole `sessions/<sid>/` directory is removed with one `DELETE /api/2.0/fs/directories` (replaces the old orphan-chunk sweep).
* **Metrics** — per-stage latency histograms (manifest / PUT-stream / complete), `files_api_429_total`, `staging_dir_count` (janitor efficacy), `event_backlog` — the backpressure signals from the Resilience Addendum, exposed at `/metrics`.
* **Correlation** — `correlation_id = file_id` on every event links the complete trail (part → registry → lifecycle → event) for the ops board.

## 6. Verification — end-to-end smoke (dev)

```bash
# 1) manifest — expect 201 + per-file upload URLs
Ses=$(curl -s -X POST $API/upload-sessions -H "Authorization: Bearer $TOKEN" \
  -d '{"file_type_code":"order", "files":[{"original_name":"orders_daily.csv","declared_size":1024}]}')

# 2) streamed upload — expect 200 + sha256 + finalized:false
Fid=$(echo $Ses | jq -r '.files[0].file_id')
curl -s -X PUT "$API/upload-sessions/$Ses/files/$Fid" \
     -H "Authorization: Bearer $TOKEN" --data-binary @orders_daily.csv

# 3) complete — expect COMPLETED + finalized:[file_id]
curl -s -X POST "$API/upload-sessions/$Ses/complete" -H "Authorization: Bearer $TOKEN"

# 4) negative path — same file again: expect duplicates:[file_id], status COMPLETED_PARTIAL
curl -s -X POST "$API/upload-sessions/new_session/complete" …
```

Then assert in the workspace (this part is runnable here — Databricks-side checks):

```python
from pathlib import Path
staged = Path("/Volumes/mdd_platform/fvr/staging/sessions")
print("session dirs:", [p.name for p in staged.iterdir()])   # expect ["ses_<sid>"] after step 3
```

… and in `mongosh`: `upload_sessions` shows `COMPLETED`; `file_registry` row is `RECEIVED` with `staged_path` set and `gridfs_file_id: null`; `file_lifecycle` first entry is `STAGING_VOLUME`; `event_log` carries exactly one `FILE_ARRIVED` whose `dedup_key = file_id`.

**Step 3 is DONE when:** the smoke passes end-to-end, the duplicate path returns the original `file_id`, and re-running any step produces no duplicate Mongo artifacts. Validation + routing then land in Step 4, and everything is staged for the **R2 checkpoint — rule dry-run + load test** (companion notebook).

## 7. Code review (2026-11-09) — findings & fixes applied

| # | Severity | Finding | Fix applied |
| --- | --- | --- | --- |
| F1 | **blocker** | `asyncio` used in `_retry` but never imported — `NameError` on the first 429/5xx retry; transport faults (`httpx.RequestError`) were not retried at all, only status codes | import added; retry loop wraps the request in `try/except httpx.RequestError` → backoff + continue |
| F2 | **blocker** | `except mongo.DuplicateKeyError` — motor does not re-export that name; duplicate detection would raise `AttributeError` and 500 instead of a clean 409 | imports `from pymongo.errors import DuplicateKeyError`; handler corrected |
| F3 | **major** | raising `HTTPException(413)` **inside** the async generator consumed by httpx does not map to a clean 413 — it surfaces as an opaque send failure and can wedge the exchange mid-PUT | flag-based abort: generator sets `cap_exceeded` and returns; endpoint then **deletes the partial staged object** and raises 413 |
| F4 | **major** | `complete` had no `IN_PROGRESS` state guard — re-completing a finished session would re-run finalize, hit the sha256 duplicate index, and mislabel the session `COMPLETED_PARTIAL`; the terminal update was likewise unguarded | find filtered on `status: IN_PROGRESS` (409 otherwise); progress + terminal updates both guarded on `IN_PROGRESS` |
| F5 | minor | `datetime.utcnow()` deprecated (Python 3.12+) | all occurrences → `datetime.now(timezone.utc)` |
| F6 | minor | `parts.$.sha256` written by the upload endpoint but not declared in the Step-2 `$jsonSchema` for `upload_sessions.parts` | schema now declares `sha256` (string/null, ≤64) — Step-2 notebook updated in the same commit |

**Non-blocking notes (verify at smoke, not code changes):** the finalize HEAD check assumes the Files API returns `Content-Length` on `HEAD /fs/files` (confirm in step 6.4 of cell 6); `extension` derivation assumes a dot in the filename — guaranteed because `pre_staging_checks` rejects unknown extensions first; `pre_staging_checks` is referenced, not defined here — it lives in `domain/rules_pre.py` and reads `file_types`/`validation_rules` from the registry (rules-as-data, no hard-coded logic).

**Verdict:** with F1–F4 fixed the API is sound for dev deployment; the guarded-atomic discipline (state guard, unique indexes, ack-after-durable-write) now covers every write path.